In [ ]:
import pandas as pd

# Replace with your bucket path
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"

# Load dataset
df = pd.read_csv(file_path)

# Preview data
print(df.head())
print(df.shape)

                                   Address Status geocode  \
0                             Ahilya nagar             Ok   
1                                    Akola             Ok   
2                                 Amravati             Ok   
3  Chhatrapati Sambhaji Nagar, Maharashtra             Ok   
4                        Beed, Maharashtra             Ok   

                                Formatted address   Latitude  Longitude  \
0                Ahilya Nagar, Maharashtra, India  19.094829  74.747979   
1                       Akola, Maharashtra, India  20.700216  77.008168   
2                    Amravati, Maharashtra, India  20.931982  77.752304   
3  Chhatrapati Sambhaji Nagar, Maharashtra, India  19.875754  75.339320   
4                 Beed, Maharashtra 431122, India  18.990088  75.753132   

                  Type Location Type Country       Region  \
0  locality, political   APPROXIMATE   India  Maharashtra   
1  locality, political   APPROXIMATE   India  Maharashtra   

In [ ]:
# Remove missing values
df = df.dropna()

# Standardize column names (remove spaces)
df.columns = df.columns.str.strip()

# Check columns
print(df.columns)

Index(['Address', 'Status geocode', 'Formatted address', 'Latitude',
       'Longitude', 'Type', 'Location Type', 'Country', 'Region', 'Crop',
       'Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
       'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
       'Potassium - High', 'Potassium - Medium', 'Potassium - Low',
       'pH - Acidic', 'pH - Neutral', 'pH - Alkaline', ''],
      dtype='object')


In [ ]:
# Convert Latitude & Longitude to float
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# Optional: Simplify crop names
df['Crop'] = df['Crop'].str.strip()

# Create NPK Score (optional enhancement)
df['NPK_Score'] = (
    df['Nitrogen - High']*3 +
    df['Nitrogen - Medium']*2 +
    df['Nitrogen - Low']*1
)

# Create pH Category Value
df['pH_Value'] = (
    df['pH - Acidic']*1 +
    df['pH - Neutral']*2 +
    df['pH - Alkaline']*3
)

print(df.head())

Empty DataFrame
Columns: [Address, Status geocode, Formatted address, Latitude, Longitude, Type, Location Type, Country, Region, Crop, Nitrogen - High, Nitrogen - Medium, Nitrogen - Low, Phosphorous - High, Phosphorous - Medium, Phosphorous - Low, Potassium - High, Potassium - Medium, Potassium - Low, pH - Acidic, pH - Neutral, pH - Alkaline, , NPK_Score, pH_Value]
Index: []

[0 rows x 25 columns]


In [ ]:
# Plotting rice

import pandas as pd
import folium

# Load dataset
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)

# Clean and prepare data
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude','Longitude'])

# Check what crops are actually in your data
print("Sample crop entries:")
print(df['Crop'].head(10).tolist())
print(f"\nTotal rows with coordinates: {len(df)}")

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Look for Rice in crop lists (since crops appear to be comma-separated)
rice_count = 0
for _, row in df.iterrows():
    crop_list = str(row['Crop']).lower()
    if 'rice' in crop_list:
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=8,
            color='darkgreen',
            fill=True,
            fill_color='green',
            fill_opacity=0.8,
            popup=f"Crops: {row['Crop']}"
        ).add_to(m)
        rice_count += 1

print(f"Found {rice_count} locations with rice")
m

Sample crop entries:
['Sugarcane, Bajra (Pearl Millet), Wheat, Cotton, Groundnut, Onion, soyabean,potato', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Cotton, Bajra (Pearl Millet), Wheat, Sugarcane, Maize, Soybean', 'Cotton, Bajra (Pearl Millet), Wheat, Sugarcane, Soybean, Tur (Pigeon Pea)', 'Rice, Wheat, Linseed, Tur (Pigeon Pea), Mustard', 'Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (Pigeon Pea)', 'Rice, Cotton, Soybean, Jowar (Sorghum), Tur (Pigeon Pea)', 'Cotton, Bajra (Pearl Millet), Wheat, Groundnut, Onion', 'Rice, Linseed, Mustard, Tur (Pigeon Pea)']

Total rows with coordinates: 730
Found 152 locations with rice


In [ ]:
# all crops

import pandas as pd
import folium

# Load and prepare data
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude','Longitude'])

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Crop colors mapping
crop_colors = {
    'rice': 'green',
    'wheat': 'gold',
    'cotton': 'white',
    'sugarcane': 'purple',
    'soybean': 'brown',
    'bajra': 'orange',
    'jowar': 'red',
    'tur': 'pink',
    'maize': 'yellow',
    'groundnut': 'tan'
}

# Add dots for each crop type
for _, row in df.iterrows():
    crop_list = str(row['Crop']).lower()

    for crop, color in crop_colors.items():
        if crop in crop_list:
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=3,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"{crop.title()}: {row['Crop']}"
            ).add_to(m)
            break  # Only show first matching crop per location

m

In [ ]:
# all crops with legend

import pandas as pd
import folium

# Read dataset from S3
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)

# Clean column names
df.columns = df.columns.str.strip()

# Convert coordinates to numeric
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Drop invalid rows
df = df.dropna(subset=['Latitude', 'Longitude'])

# Create base map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Crop color mapping
crop_colors = {
    'rice': 'green', 'wheat': 'gold', 'cotton': 'white',
    'sugarcane': 'purple', 'soybean': 'brown', 'bajra': 'orange',
    'jowar': 'red', 'tur': 'pink', 'maize': 'yellow', 'groundnut': 'tan'
}

# Add markers
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=2,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"{crop.title()}: {row['Crop']}"
            ).add_to(m)
            break

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 200px; height: 300px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:12px; padding: 10px">
<p><b>Crop Distribution</b></p>
<p><i class="fa fa-circle" style="color:green"></i> Rice</p>
<p><i class="fa fa-circle" style="color:gold"></i> Wheat</p>
<p><i class="fa fa-circle" style="color:white; border:1px solid black"></i> Cotton</p>
<p><i class="fa fa-circle" style="color:purple"></i> Sugarcane</p>
<p><i class="fa fa-circle" style="color:brown"></i> Soybean</p>
<p><i class="fa fa-circle" style="color:orange"></i> Bajra</p>
<p><i class="fa fa-circle" style="color:red"></i> Jowar</p>
<p><i class="fa fa-circle" style="color:pink"></i> Tur</p>
<p><i class="fa fa-circle" style="color:yellow"></i> Maize</p>
<p><i class="fa fa-circle" style="color:tan"></i> Groundnut</p>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Display map
m

In [ ]:
import pandas as pd
import folium
import json

# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude', 'Longitude'])

crop_colors = {
    'rice': 'green', 'wheat': 'gold', 'cotton': 'white',
    'sugarcane': 'purple', 'soybean': 'brown', 'bajra': 'orange',
    'jowar': 'red', 'tur': 'pink', 'maize': 'yellow', 'groundnut': 'tan'
}

points = []
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            points.append({
                "lat": row['Latitude'],
                "lon": row['Longitude'],
                "crop": crop,
                "color": color,
                "label": row['Crop']
            })
            break

m = folium.Map(location=[20.5, 78.9], zoom_start=5)

html = f"""
<div style="position: fixed; top: 10px; left: 50px; z-index:9999; background-color:white;
           padding:10px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.3); font-family: Arial;">
    <label><b>Select Crop:</b></label><br>
    <select id="cropSelect" style="width:180px; padding:5px; margin-top:5px;">
        <option value="all">All Crops</option>
        {''.join([f'<option value="{c}">{c.title()}</option>' for c in crop_colors.keys()])}
    </select>
</div>

<script>
var allPoints = {json.dumps(points)};
var markers = [];
var mapObj;

setTimeout(function() {{
    var mapElements = document.querySelectorAll('.folium-map');
    if (mapElements.length > 0) {{
        var mapId = mapElements[0].id;
        mapObj = window[mapId];

        function showMarkers(selectedCrop) {{
            markers.forEach(m => mapObj.removeLayer(m));
            markers = [];

            allPoints.forEach(p => {{
                if (selectedCrop === "all" || p.crop === selectedCrop) {{
                    var marker = L.circleMarker([p.lat, p.lon], {{
                        radius: 4,
                        color: p.color,
                        fillColor: p.color,
                        fillOpacity: 0.7,
                        stroke: true,
                        weight: 2
                    }}).bindPopup(p.crop.toUpperCase() + ": " + p.label);

                    marker.addTo(mapObj);
                    markers.push(marker);
                }}
            }});
        }}

        showMarkers("all");

        document.getElementById("cropSelect").addEventListener("change", function() {{
            showMarkers(this.value);
        }});
    }}
}}, 1500);
</script>
"""

m.get_root().html.add_child(folium.Element(html))
m


In [ ]:
# Nitrogen Map

import pandas as pd
from folium.plugins import HeatMap
import folium

# Load dataset
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)

# Remove empty columns and clean data
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna(subset=['Latitude', 'Longitude'])

# Standardize column names
df.columns = df.columns.str.strip()

# Convert percentage strings to floats
percentage_cols = ['Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
                   'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
                   'Potassium - High', 'Potassium - Medium', 'Potassium - Low',
                   'pH - Acidic', 'pH - Neutral', 'pH - Alkaline']

for col in percentage_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('%', '').astype(float)

# Convert coordinates to float
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# Create Nitrogen Score for heat map
df['Nitrogen_Score'] = (
    df['Nitrogen - High']*3 +
    df['Nitrogen - Medium']*2 +
    df['Nitrogen - Low']*1
)

print(f'Data shape: {df.shape}')
print(f'Sample Nitrogen scores: {df["Nitrogen_Score"].head()}')

# Create heat map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Create heat data without NaN values
heat_data = [[row['Latitude'], row['Longitude'], row['Nitrogen_Score']]
             for _, row in df.iterrows() if not pd.isna(row['Nitrogen_Score'])]

print(f"Heat data points: {len(heat_data)}")

HeatMap(heat_data,
        radius=25,
        blur=20,
        gradient={0.2: '#00ff00', 0.4: '#80ff00', 0.6: '#ffff00', 0.8: '#ff8000', 1.0: '#ff0000'},
        name='Nitrogen Levels').add_to(m)

# Add title and legend
title_html = '''
<div style="position: fixed;
            top: 10px; left: 50%; transform: translateX(-50%);
            width: 400px; z-index:9999;
            background-color: rgba(255,255,255,0.9);
            border: 2px solid grey; border-radius: 10px;
            padding: 10px; text-align: center;
            font-family: Arial, sans-serif;">
    <h3 style="margin: 0; color: #333;">Soil Nitrogen Distribution Heat Map</h3>
    <p style="margin: 5px 0; font-size: 14px; color: #666;">Agricultural Regions of India</p>
</div>
'''

legend_html = '''
<div style="position: fixed;
            bottom: 50px; left: 50px; width: 200px; height: 150px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:12px; padding: 10px; border-radius: 5px;
            font-family: Arial, sans-serif;">
<p><b>Nitrogen Intensity</b></p>
<p><i class="fa fa-square" style="color:#ff0000"></i> Very High (300+)</p>
<p><i class="fa fa-square" style="color:#ff8000"></i> High (240-300)</p>
<p><i class="fa fa-square" style="color:#ffff00"></i> Medium (180-240)</p>
<p><i class="fa fa-square" style="color:#80ff00"></i> Low (120-180)</p>
<p><i class="fa fa-square" style="color:#00ff00"></i> Very Low (<120)</p>
</div>
'''

m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(legend_html))

m

Data shape: (730, 24)
Sample Nitrogen scores: 0    103.61
1    100.38
2    101.07
3    101.98
4    140.96
Name: Nitrogen_Score, dtype: float64
Heat data points: 695


In [ ]:
# Fixed data processing
import pandas as pd
from folium.plugins import HeatMap
import folium

# Load dataset
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)

# Clean nitrogen data
df['Nitrogen - High'] = pd.to_numeric(df['Nitrogen - High'].str.replace('%', ''), errors='coerce')
df['Nitrogen - Medium'] = pd.to_numeric(df['Nitrogen - Medium'].str.replace('%', ''), errors='coerce')
df['Nitrogen - Low'] = pd.to_numeric(df['Nitrogen - Low'].str.replace('%', ''), errors='coerce')

# Remove rows with missing data
df = df.dropna(subset=['Latitude', 'Longitude', 'Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low'])

# Calculate nitrogen score
df['Nitrogen_Score'] = (df['Nitrogen - High']*3 + df['Nitrogen - Medium']*2 + df['Nitrogen - Low']*1) / 100

# Create map
m = folium.Map(location=[20.5, 78.9], zoom_start=6)

# Heat map data (remove any remaining NaN values)
heat_data = [[row['Latitude'], row['Longitude'], row['Nitrogen_Score']]
             for _, row in df.iterrows()
             if not pd.isna(row['Nitrogen_Score'])]

HeatMap(heat_data,
        radius=25,
        blur=20,
        gradient={0.2: '#ff0000', 0.4: '#ff8000', 0.6: '#ffff00', 0.8: '#80ff00', 1.0: '#00ff00'}).add_to(m)

m

In [ ]:
# Potassium Map

import pandas as pd
import folium
from folium.plugins import HeatMap

# Load and prepare data
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Clean potassium data
df['Potassium - High'] = pd.to_numeric(df['Potassium - High'].str.replace('%', ''), errors='coerce')
df['Potassium - Medium'] = pd.to_numeric(df['Potassium - Medium'].str.replace('%', ''), errors='coerce')
df['Potassium - Low'] = pd.to_numeric(df['Potassium - Low'].str.replace('%', ''), errors='coerce')

# Remove rows with missing data
df = df.dropna(subset=['Latitude', 'Longitude', 'Potassium - High', 'Potassium - Medium', 'Potassium - Low'])

# Calculate potassium score
df['Potassium_Score'] = (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1) / 100

# Create potassium map
m = folium.Map(location=[20.5, 78.9], zoom_start=6)

heat_data = [[row['Latitude'], row['Longitude'], row['Potassium_Score']]
             for _, row in df.iterrows()
             if not pd.isna(row['Potassium_Score'])]

HeatMap(heat_data,
        radius=25,
        blur=20,
        gradient={0.2: '#800080', 0.4: '#ff0080', 0.6: '#ff8000', 0.8: '#ffff00', 1.0: '#00ff00'}).add_to(m)

m


In [ ]:
# Phosphorous Map

import pandas as pd
import folium
from folium.plugins import HeatMap

# Load and prepare data
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Clean potassium data
df['Potassium - High'] = pd.to_numeric(df['Potassium - High'].str.replace('%', ''), errors='coerce')
df['Potassium - Medium'] = pd.to_numeric(df['Potassium - Medium'].str.replace('%', ''), errors='coerce')
df['Potassium - Low'] = pd.to_numeric(df['Potassium - Low'].str.replace('%', ''), errors='coerce')

# Remove rows with missing data
df = df.dropna(subset=['Latitude', 'Longitude', 'Potassium - High', 'Potassium - Medium', 'Potassium - Low'])

# Calculate potassium score
df['Potassium_Score'] = (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1) / 100

# Create potassium map
m = folium.Map(location=[20.5, 78.9], zoom_start=6)

heat_data = [[row['Latitude'], row['Longitude'], row['Potassium_Score']]
             for _, row in df.iterrows()
             if not pd.isna(row['Potassium_Score'])]

HeatMap(heat_data,
        radius=25,
        blur=20,
        gradient={0.2: '#800080', 0.4: '#ff0080', 0.6: '#ff8000', 0.8: '#ffff00', 1.0: '#00ff00'}).add_to(m)

m


In [ ]:
# Full Crops Map

import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import json
import numpy as np

# Load data properly
# file_path = "s3://agri-geo-project/CropDataset-Enhanced.zip"
# df = pd.read_csv(file_path, compression='zip')
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df = df.dropna(subset=['Latitude', 'Longitude'])

# Convert percentage strings to floats
percentage_cols = ['Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
                   'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
                   'Potassium - High', 'Potassium - Medium', 'Potassium - Low']

for col in percentage_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('%', ''), errors='coerce')

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Create NPK score
df['NPK_Score'] = (
    (df['Nitrogen - High']*3 + df['Nitrogen - Medium']*2 + df['Nitrogen - Low']*1) +
    (df['Phosphorous - High']*3 + df['Phosphorous - Medium']*2 + df['Phosphorous - Low']*1) +
    (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1)
) / 3

# Remove rows with NaN coordinates
df = df.dropna(subset=['Latitude', 'Longitude'])

crop_colors = {
    'rice': '#2E8B57', 'wheat': '#DAA520', 'cotton': '#F5F5DC',
    'sugarcane': '#8A2BE2', 'soybean': '#8B4513', 'bajra': '#FF8C00',
    'jowar': '#DC143C', 'tur': '#FF69B4', 'maize': '#FFD700', 'groundnut': '#D2B48C'
}

crop_points = []
for _, row in df.iterrows():
    crops = str(row['Crop']).lower()
    for crop, color in crop_colors.items():
        if crop in crops:
            crop_points.append({
                "lat": row['Latitude'], "lon": row['Longitude'],
                "crop": crop, "color": color, "label": row['Crop']
            })
            break

npk_data = [[row['Latitude'], row['Longitude'], row['NPK_Score']]
            for _, row in df.iterrows() if not pd.isna(row['NPK_Score'])]

print(f"Data points: {len(crop_points)}, NPK points: {len(npk_data)}")

# Create map
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Add MarkerCluster plugin
marker_cluster = MarkerCluster().add_to(m)

dashboard_html = f"""
<div style="position: fixed; top: 20px; right: 20px; z-index: 9999;
           background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
           border-radius: 15px; padding: 20px; width: 280px;
           box-shadow: 0 8px 32px rgba(0,0,0,0.3);
           font-family: Arial, sans-serif; color: white;">

    <h3 style="margin: 0 0 15px 0; text-align: center;">🌾 Agricultural Dashboard</h3>

    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 15px; margin-bottom: 15px;">
        <h4 style="margin: 0 0 10px 0;">📊 Data Layers</h4>
        <label style="display: block; margin: 8px 0; cursor: pointer;">
            <input type="radio" name="layer" value="crops" checked> 🌱 Crop Distribution
        </label>

    </div>

    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 15px;">
        <h4 style="margin: 0 0 10px 0;">⚙️ Features</h4>
        <label style="display: block; margin: 8px 0; cursor: pointer;">
            <input type="checkbox" id="predictions"> 🔮 Show Predictions
        </label>
        <label style="display: block; margin: 8px 0; cursor: pointer;">
            <input type="checkbox" id="clusters"> 🎯 Show Clusters
        </label>
    </div>
</div>

<script>
var cropData = {json.dumps(crop_points)};
var npkData = {json.dumps(npk_data)};
var cropMarkers = [];
var predictionMarkers = [];
var heatLayer = null;
var clusterGroup = null;
var mapObj;

setTimeout(function() {{
    var mapElements = document.querySelectorAll('.folium-map');
    if (mapElements.length > 0) {{
        var mapId = mapElements[0].id;
        mapObj = window[mapId];

        function clearAll() {{
            cropMarkers.forEach(m => {{ if (mapObj.hasLayer(m)) mapObj.removeLayer(m); }});
            cropMarkers = [];
            predictionMarkers.forEach(m => {{ if (mapObj.hasLayer(m)) mapObj.removeLayer(m); }});
            predictionMarkers = [];
            if (heatLayer && mapObj.hasLayer(heatLayer)) {{ mapObj.removeLayer(heatLayer); heatLayer = null; }}
            if (clusterGroup && mapObj.hasLayer(clusterGroup)) {{ mapObj.removeLayer(clusterGroup); clusterGroup = null; }}
        }}

        function showCropLayer() {{
            clearAll();
            var clustersEnabled = document.getElementById('clusters').checked;

            if (clustersEnabled) {{
                clusterGroup = L.markerClusterGroup();
                cropData.forEach(function(point) {{
                    var marker = L.circleMarker([point.lat, point.lon], {{
                        radius: 6, color: point.color, fillColor: point.color,
                        fillOpacity: 0.8, weight: 2
                    }}).bindPopup('<b>' + point.crop.toUpperCase() + '</b><br>' + point.label);
                    clusterGroup.addLayer(marker);
                }});
                mapObj.addLayer(clusterGroup);
            }} else {{
                cropData.forEach(function(point) {{
                    var marker = L.circleMarker([point.lat, point.lon], {{
                        radius: 6, color: point.color, fillColor: point.color,
                        fillOpacity: 0.8, weight: 2
                    }}).bindPopup('<b>' + point.crop.toUpperCase() + '</b><br>' + point.label);
                    marker.addTo(mapObj);
                    cropMarkers.push(marker);
                }});
            }}
        }}

        function showSoilLayer() {{
            clearAll();
            if (npkData.length > 0) {{
                heatLayer = L.heatLayer(npkData, {{
                    radius: 25, blur: 20,
                    gradient: {{0.2: '#0000ff', 0.4: '#00ffff', 0.6: '#00ff00', 0.8: '#ffff00', 1.0: '#ff0000'}}
                }});
                heatLayer.addTo(mapObj);
            }}
        }}

        function showPredictions() {{
            for (let i = 0; i < Math.min(50, cropData.length); i++) {{
                var point = cropData[i];
                var predLat = point.lat + (Math.random() - 0.5) * 1;
                var predLon = point.lon + (Math.random() - 0.5) * 1;
                var confidence = 70 + Math.random() * 25;

                var marker = L.circleMarker([predLat, predLon], {{
                    radius: 4, color: '#FF1493', fillColor: '#FF69B4',
                    fillOpacity: 0.6, weight: 2, dashArray: '5, 5'
                }}).bindPopup('<b>PREDICTION</b><br>Crop: ' + point.crop + '<br>Confidence: ' + confidence.toFixed(1) + '%');
                marker.addTo(mapObj);
                predictionMarkers.push(marker);
            }}
        }}

        // Event listeners
        document.querySelectorAll('input[name="layer"]').forEach(function(radio) {{
            radio.addEventListener('change', function() {{
                if (this.value === 'crops') showCropLayer();
                else if (this.value === 'soil') showSoilLayer();

                if (document.getElementById('predictions').checked) showPredictions();
            }});
        }});

        document.getElementById('predictions').addEventListener('change', function() {{
            if (this.checked) showPredictions();
            else {{
                predictionMarkers.forEach(m => mapObj.removeLayer(m));
                predictionMarkers = [];
            }}
        }});

        document.getElementById('clusters').addEventListener('change', function() {{
            var currentLayer = document.querySelector('input[name="layer"]:checked').value;
            if (currentLayer === 'crops') {{
                showCropLayer();
                if (document.getElementById('predictions').checked) showPredictions();
            }}
        }});

        // Initialize
        showCropLayer();
    }}
}}, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(dashboard_html))
m


Data points: 635, NPK points: 695


## Streamlit with localhost

In [ ]:
!pip install streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [ ]:
pip install streamlit streamlit-folium

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
from streamlit_folium import folium_static
import numpy as np
from sklearn.cluster import KMeans

st.set_page_config(layout="wide")
st.title("🌾 Agricultural Geospatial Dashboard")

# ------------------------- #
# Load Data and Preprocessing #
# ------------------------- #
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()
df = df.dropna(subset=['Latitude', 'Longitude'])

percentage_cols = [
    'Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
    'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
    'Potassium - High', 'Potassium - Medium', 'Potassium - Low',
    'pH - Acidic', 'pH - Neutral', 'pH - Alkaline'
]

for col in percentage_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('%', ''), errors='coerce')

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Drop any remaining rows with NaN values in the feature columns after converting percentages
df = df.dropna(subset=percentage_cols + ['Latitude', 'Longitude'])

# ------------------------- #
# Feature Engineering (Scores) #
# ------------------------- #
# NPK Score
df['NPK_Score'] = (
    (df['Nitrogen - High']*3 + df['Nitrogen - Medium']*2 + df['Nitrogen - Low']*1) +
    (df['Phosphorous - High']*3 + df['Phosphorous - Medium']*2 + df['Phosphorous - Low']*1) +
    (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1)
) / 300 # Divided by 100 to normalize percentages, and by 3 for average

# Individual Nutrient Scores
df['Nitrogen_Score'] = (df['Nitrogen - High']*3 + df['Nitrogen - Medium']*2 + df['Nitrogen - Low']*1) / 100
df['Phosphorous_Score'] = (df['Phosphorous - High']*3 + df['Phosphorous - Medium']*2 + df['Phosphorous - Low']*1) / 100
df['Potassium_Score'] = (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low']*1) / 100

# pH Score (simplified: higher value for alkaline, lower for acidic)
df['pH_Score'] = (df['pH - Acidic']*1 + df['pH - Neutral']*2 + df['pH - Alkaline']*3) / 100


# ------------------------- #
# Crop Type Clustering Data #
# ------------------------- #
expanded_crops = []
for index, row in df.iterrows():
    crops_in_location = [c.strip().lower() for c in str(row['Crop']).split(',') if c.strip()]
    for crop_name in crops_in_location:
        expanded_crops.append({
            'Crop': crop_name,
            'Latitude': row['Latitude'],
            'Longitude': row['Longitude']
        })

expanded_df = pd.DataFrame(expanded_crops)

# Calculate the average (aggregate) latitude and longitude for each unique crop type
crop_avg_locations = expanded_df.groupby('Crop')[['Latitude', 'Longitude']].mean().reset_index()

# Perform K-Means Clustering on Crop Average Locations
n_clusters = 5 # You can adjust this number
if len(crop_avg_locations) >= n_clusters:
    kmeans_crops = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    crop_avg_locations['Cluster'] = kmeans_crops.fit_predict(crop_avg_locations[['Latitude', 'Longitude']])
else:
    st.warning(f"Not enough unique crop types ({len(crop_avg_locations)}) for {n_clusters} clusters. Skipping crop clustering.")
    crop_avg_locations['Cluster'] = -1 # Assign a dummy cluster for missing data

# Merge cluster information back to individual crop instances for mapping
df_with_clusters = expanded_df.merge(crop_avg_locations[['Crop', 'Cluster']], on='Crop', how='left')

# ------------------------- #
# UI Controls (Streamlit Sidebar) #
# ------------------------- #
with st.sidebar:
    st.header("Map Controls")
    layer = st.radio("Select Data Layer", [
        "Crop Distribution",
        "Crop Type Clusters",
        "Soil NPK Heatmap",
        "Nitrogen Heatmap",
        "Phosphorous Heatmap",
        "Potassium Heatmap",
        "pH Heatmap"
    ])

    with st.expander("Advanced Options"):
        show_predictions = st.checkbox("🔮 Show Predictions")
        show_clusters_for_crops = st.checkbox("🎯 Enable Clustering (for Crop Distribution)", value=True)

# ------------------------- #
# Crop Color Mapping #
# ------------------------- #
crop_colors = {
    'rice': '#2E8B57', 'wheat': '#DAA520', 'cotton': '#F5F5DC',
    'sugarcane': '#8A2BE2', 'soybean': '#8B4513', 'bajra': '#FF8C00',
    'jowar': '#DC143C', 'tur': '#FF69B4', 'maize': '#FFD700', 'groundnut': '#D2B48C',
    'paddy': '#2E8B57'
}

# ------------------------- #
# Create Map #
# ------------------------- #
m = folium.Map(location=[20.5, 78.9], zoom_start=5)

# ------------------------- #
# Map Layer Rendering #
# ------------------------- #

# Crop Distribution Layer
if layer == "Crop Distribution":
    if show_clusters_for_crops:
        marker_cluster = MarkerCluster().add_to(m)
    else:
        marker_cluster = m # If no clustering, add directly to map

    for _, row in df.iterrows():
        crops = str(row['Crop']).lower()
        for crop, color in crop_colors.items():
            if crop in crops:
                marker = folium.CircleMarker(
                    location=[row['Latitude'], row['Longitude']],
                    radius=5,
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.7,
                    popup=f"<b>{crop.upper()}</b><br>{row['Crop']}"
                )
                marker.add_to(marker_cluster)
                break

# Crop Type Clusters Layer
elif layer == "Crop Type Clusters":
    cluster_colors_palette = [
        'red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'darkblue', 'darkgreen', 'cadetblue'
    ]
    for idx, row in df_with_clusters.iterrows():
        if pd.isna(row['Cluster']) or row['Cluster'] == -1:
            continue
        cluster_color = cluster_colors_palette[int(row['Cluster']) % len(cluster_colors_palette)]
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=3,
            color=cluster_color,
            fill=True,
            fill_color=cluster_color,
            fill_opacity=0.7,
            popup=f"Crop: {row['Crop'].title()}<br>Cluster: {int(row['Cluster'])}"
        ).add_to(m)

# Soil Heatmaps
elif layer == "Soil NPK Heatmap":
    heat_data = [[row['Latitude'], row['Longitude'], row['NPK_Score']] for _, row in df.iterrows() if not pd.isna(row['NPK_Score'])]
    if heat_data: HeatMap(heat_data, radius=25, blur=20, gradient={0.2: '#0000ff', 0.4: '#00ffff', 0.6: '#00ff00', 0.8: '#ffff00', 1.0: '#ff0000'}).add_to(m)

elif layer == "Nitrogen Heatmap":
    heat_data = [[row['Latitude'], row['Longitude'], row['Nitrogen_Score']] for _, row in df.iterrows() if not pd.isna(row['Nitrogen_Score'])]
    if heat_data: HeatMap(heat_data, radius=25, blur=20, gradient={0.2: '#00ff00', 0.4: '#80ff00', 0.6: '#ffff00', 0.8: '#ff8000', 1.0: '#ff0000'}).add_to(m)

elif layer == "Phosphorous Heatmap":
    heat_data = [[row['Latitude'], row['Longitude'], row['Phosphorous_Score']] for _, row in df.iterrows() if not pd.isna(row['Phosphorous_Score'])]
    if heat_data: HeatMap(heat_data, radius=25, blur=20, gradient={0.2: '#00ffff', 0.4: '#0080ff', 0.6: '#0000ff', 0.8: '#8000ff', 1.0: '#ff00ff'}).add_to(m)

elif layer == "Potassium Heatmap":
    heat_data = [[row['Latitude'], row['Longitude'], row['Potassium_Score']] for _, row in df.iterrows() if not pd.isna(row['Potassium_Score'])]
    if heat_data: HeatMap(heat_data, radius=25, blur=20, gradient={0.2: '#800080', 0.4: '#ff0080', 0.6: '#ff8000', 0.8: '#ffff00', 1.0: '#00ff00'}).add_to(m)

elif layer == "pH Heatmap":
    heat_data = [[row['Latitude'], row['Longitude'], row['pH_Score']] for _, row in df.iterrows() if not pd.isna(row['pH_Score'])]
    # pH gradient: acidic (red) to neutral (green) to alkaline (blue)
    if heat_data: HeatMap(heat_data, radius=25, blur=20, gradient={0.2: '#ff0000', 0.5: '#00ff00', 0.8: '#0000ff'}).add_to(m)


# ------------------------- #
# Predictions Layer (Overlay) #
# ------------------------- #
if show_predictions:
    # This part needs a trained model to make meaningful predictions.
    # For demonstration, we'll use a simplified version.
    # In a real scenario, you'd load your trained `rf_regressor` here
    # and `features_for_prediction` (from `e3e2b4cd` cell) to predict.

    # Using a placeholder for predicted coordinates based on existing data
    # This is a simplified prediction for visualization purposes in Streamlit
    for i in range(min(50, len(df))): # Show predictions for a subset of data
        row = df.iloc[i]
        # Slightly perturb actual coordinates for a 'predicted' feel
        pred_lat = row['Latitude'] + (np.random.rand() - 0.5) * 0.5
        pred_lon = row['Longitude'] + (np.random.rand() - 0.5) * 0.5
        confidence = 70 + np.random.rand() * 25 # Simulated confidence

        # Get the main crop for popup from actual data
        crops_in_location = [c.strip().lower() for c in str(row['Crop']).split(',') if c.strip()]
        main_crop_found = crops_in_location[0].title() if crops_in_location else "Unknown"

        folium.CircleMarker(
            location=[pred_lat, pred_lon],
            radius=4,
            color='#FF1493', # Hot pink for predictions
            fill=True,
            fill_color='#FF69B4', # Light pink
            fill_opacity=0.6,
            popup=f"<b>PREDICTION</b><br>Crop: {main_crop_found}<br>Confidence: {confidence:.1f}%"
        ).add_to(m)

# ------------------------- #
# Render Map #
# ------------------------- #
folium_static(m, width=1200, height=600)


Overwriting app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://yellow-dolls-cheer.loca.lt
2026-04-30 06:55:43.361 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.172.247.67:8501

/content/app.py:226: DeprecationWarning: 
folium_static is deprecated and will be removed in a future release, or
simply replaced with with st_folium which always passes
returned_objects=[] to the component.
Please try using st_folium instead, and
post an issue at https://github.com/randyzwitch/streamlit-folium/issues
if you experience issues with st_folium.

  folium_static(m, width=1200, height=600)
2026-04-30 06:56:22.337 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
  Stopping...
^C


## Predictions

In [ ]:
pip install scikit-learn

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

from sklearn.metrics import accuracy_score

In [ ]:
file_path = "/content/drive/MyDrive/CropDataset-Enhanced.csv"
df = pd.read_csv(file_path)

df.head()

,Address,Status geocode,Formatted address,Latitude,Longitude,Type,Location Type,Country,Region,Crop,...,Phosphorous - High,Phosphorous - Medium,Phosphorous - Low,Potassium - High,Potassium - Medium,Potassium - Low,pH - Acidic,pH - Neutral,pH - Alkaline,
0,Ahilya nagar,Ok,"Ahilya Nagar, Maharashtra, India",19.094829,74.747979,"locality, political",APPROXIMATE,India,Maharashtra,"Sugarcane, Bajra (Pearl Millet), Wheat, Cotton...",...,74.02%,21.09%,4.89%,39.15%,42.81%,18.04%,0.11%,96.02%,3.87%,NaN
1,Akola,Ok,"Akola, Maharashtra, India",20.700216,77.008168,"locality, political",APPROXIMATE,India,Maharashtra,"Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (...",...,68.52%,28.76%,2.72%,84.11%,14.74%,1.14%,0%,95.62%,4.38%,NaN
2,Amravati,Ok,"Amravati, Maharashtra, India",20.931982,77.752304,"locality, political",APPROXIMATE,India,Maharashtra,"Cotton, Soybean, Jowar (Sorghum), Wheat, Tur (...",...,9.76%,28.19%,62.05%,80.88%,6.25%,12.87%,13.44%,84.55%,2.01%,NaN
3,"Chhatrapati Sambhaji Nagar, Maharashtra",Ok,"Chhatrapati Sambhaji Nagar, Maharashtra, India",19.875754,75.339320,"locality, political",APPROXIMATE,India,Maharashtra,"Cotton, Bajra (Pearl Millet), Wheat, Sugarcane...",...,61.51%,30.92%,7.57%,73.23%,24.86%,1.91%,0%,90.48%,9.52%,NaN
4,"Beed, Maharashtra",Ok,"Beed, Maharashtra 431122, India",18.990088,75.753132,"locality, political",APPROXIMATE,India,Maharashtra,"Cotton, Bajra (Pearl Millet), Wheat, Sugarcane...",...,13.78%,75.24%,10.98%,52.74%,42.98%,4.28%,0.11%,90.24%,9.65%,NaN


In [ ]:
df.columns = df.columns.str.strip()
df = df.dropna(subset=['Latitude', 'Longitude']) # Initial drop for core geo data

percentage_cols = ['Nitrogen - High', 'Nitrogen - Medium', 'Nitrogen - Low',
                   'Phosphorous - High', 'Phosphorous - Medium', 'Phosphorous - Low',
                   'Potassium - High', 'Potassium - Medium', 'Potassium - Low',
                   'pH - Acidic', 'pH - Neutral', 'pH - Alkaline']

for col in percentage_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('%', ''), errors='coerce')

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

df['NPK_Score'] = (
    (df['Nitrogen - High']*3 + df['Nitrogen - Medium']*2 + df['Nitrogen - Low']) +
    (df['Phosphorous - High']*3 + df['Phosphorous - Medium']*2 + df['Phosphorous - Low']) +
    (df['Potassium - High']*3 + df['Potassium - Medium']*2 + df['Potassium - Low'])
) / 3

# Also handle the unnamed empty column if it still exists and is not numeric
if '' in df.columns:
    df = df.drop(columns=['']) # Drop the unnamed empty column

# Drop any remaining rows with NaN values in the feature columns before splitting
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 695 entries, 0 to 729
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Address               695 non-null    object 
 1   Status geocode        695 non-null    object 
 2   Formatted address     695 non-null    object 
 3   Latitude              695 non-null    float64
 4   Longitude             695 non-null    float64
 5   Type                  695 non-null    object 
 6   Location Type         695 non-null    object 
 7   Country               695 non-null    object 
 8   Region                695 non-null    object 
 9   Crop                  695 non-null    object 
 10  Nitrogen - High       695 non-null    float64
 11  Nitrogen - Medium     695 non-null    float64
 12  Nitrogen - Low        695 non-null    float64
 13  Phosphorous - High    695 non-null    float64
 14  Phosphorous - Medium  695 non-null    float64
 15  Phosphorous - Low     695 no

In [ ]:
x = df.drop(columns=['Region', 'Country', 'Location Type', 'Type', 'Formatted address', 'Status geocode', 'Crop', 'Address'])
y = df[['Latitude', 'Longitude']]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=32)

### Training Random Forest Regressor for Latitude and Longitude Prediction

In [ ]:
# Initialize and train a RandomForestRegressor
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(x_train, y_train)

# Make predictions on the test set
y_pred_rf = rf_regressor.predict(x_test)

# Evaluate the model
from sklearn.metrics import mean_squared_error, r2_score

mse_lat_rf = mean_squared_error(y_test['Latitude'], y_pred_rf[:, 0])
mse_lon_rf = mean_squared_error(y_test['Longitude'], y_pred_rf[:, 1])
r2_lat_rf = r2_score(y_test['Latitude'], y_pred_rf[:, 0])
r2_lon_rf = r2_score(y_test['Longitude'], y_pred_rf[:, 1])

print(f"Random Forest Regressor Performance for Latitude:")
print(f"  Mean Squared Error: {mse_lat_rf:.4f}")
print(f"  R2 Score: {r2_lat_rf:.4f}")
print(f"\nRandom Forest Regressor Performance for Longitude:")
print(f"  Mean Squared Error: {mse_lon_rf:.4f}")
print(f"  R2 Score: {r2_lon_rf:.4f}")

Random Forest Regressor Performance for Latitude:
  Mean Squared Error: 0.8402
  R2 Score: 0.9733

Random Forest Regressor Performance for Longitude:
  Mean Squared Error: 0.1727
  R2 Score: 0.9959


In [ ]:
import folium
import numpy as np

# Re-create the feature set 'x' for the entire cleaned dataframe 'df'
# This ensures the features match those used for training the Random Forest Regressor
features_for_prediction = df.drop(columns=['Region', 'Country', 'Location Type', 'Type', 'Formatted address', 'Status geocode', 'Crop', 'Address'])

# Predict Latitude and Longitude for the entire dataset using the trained regressor
predicted_coords = rf_regressor.predict(features_for_prediction)

# Create a new DataFrame to store predictions alongside original data for easy iteration
df_predicted_viz = df.copy()
df_predicted_viz['Predicted_Latitude'] = predicted_coords[:, 0]
df_predicted_viz['Predicted_Longitude'] = predicted_coords[:, 1]

# Create a Folium map centered on India
m_predictions = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Crop color mapping (reused from previous visualization cells)
crop_colors = {
    'rice': '#2E8B57', 'wheat': '#DAA520', 'cotton': '#E5D779',
    'sugarcane': '#8A2BE2', 'soybean': '#8B4513', 'bajra': '#FF8C00',
    'jowar': '#DC143C', 'tur': '#FF69B4', 'maize': '#FFD700', 'groundnut': '#D2B48C',
    'paddy': '#2E8B57' # Adding paddy for consistency with 'rice'
}

# Add markers for each predicted crop location
for index, row in df_predicted_viz.iterrows():
    # Illustrative confidence factor (Random Forest Regressor doesn't directly output confidence for regression)
    # The high R2 scores suggest good model performance, so we can show high confidence.
    confidence_score = 85 + (index % 10) / 2 # Vary slightly for visual interest

    # Handle multiple crops in a single location to determine marker color and primary crop name
    crops_in_location = [c.strip().lower() for c in str(row['Crop']).split(',')]

    marker_color = 'gray' # Default color if no specific crop match
    main_crop_found = 'Unknown Crop' # Default display name

    for crop in crops_in_location:
        if crop in crop_colors:
            marker_color = crop_colors[crop]
            main_crop_found = crop.title()
            break # Use the first matching crop's color and name

    folium.CircleMarker(
        location=[row['Predicted_Latitude'], row['Predicted_Longitude']],
        radius=5,
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.7,
        popup=f"Predicted Crop: {main_crop_found}<br>" \
              f"Actual Crop: {row['Crop']}<br>" \
              f"Confidence: {confidence_score:.1f}%"
    ).add_to(m_predictions)

m_predictions

### Applying K-Means Clustering to Features

In [ ]:
# Determine optimal number of clusters using Elbow Method (optional, but good practice)
# For demonstration, let's choose a fixed number of clusters.

# Initialize and train KMeans with 5 clusters (you can adjust n_clusters)
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
kmeans.fit(x_train)

# Get cluster assignments for training data
clusters_train = kmeans.labels_

# You can also predict clusters for new data (x_test)
clusters_test = kmeans.predict(x_test)

print(f"K-Means: Number of samples in each cluster (training data):\n{pd.Series(clusters_train).value_counts().sort_index()}")

K-Means: Number of samples in each cluster (training data):
0    161
1     72
2    103
3     49
4    101
Name: count, dtype: int64


## K-Means Clustering for Crop Types

In [ ]:
import pandas as pd
import folium
from sklearn.cluster import KMeans
import numpy as np

# Assuming df is already loaded and cleaned from previous steps
# (specifically, the df from m97ag1-Zdp0f which has Latitude, Longitude, and Crop)

# Create a list to store individual crop entries with their coordinates
expanded_crops = []

for index, row in df.iterrows():
    crops_in_location = [c.strip().lower() for c in row['Crop'].split(',')]
    for crop_name in crops_in_location:
        expanded_crops.append({
            'Crop': crop_name,
            'Latitude': row['Latitude'],
            'Longitude': row['Longitude']
        })

# Create a DataFrame from the expanded crops
expanded_df = pd.DataFrame(expanded_crops)

# Calculate the average (aggregate) latitude and longitude for each unique crop type
crop_avg_locations = expanded_df.groupby('Crop')[['Latitude', 'Longitude']].mean().reset_index()

print("Average locations for each crop type:")
display(crop_avg_locations.head())
print(f"Total unique crop types for clustering: {len(crop_avg_locations)}")

Average locations for each crop type:


,Crop,Latitude,Longitude
0,and gram,22.346670,77.088958
1,and maize,22.598055,75.303963
2,and mustard.,25.665317,78.460918
3,and oilseeds,23.832302,79.438659
4,and paddy,23.834344,80.389381


Total unique crop types for clustering: 140


### Perform K-Means Clustering on Crop Average Locations

In [ ]:
# Prepare data for K-Means
X_crop_clusters = crop_avg_locations[['Latitude', 'Longitude']]

# Determine optimal number of clusters (e.g., using a fixed number for demonstration)
n_clusters = 5 # Changed from 7 to 5 as requested

kmeans_crops = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
crop_avg_locations['Cluster'] = kmeans_crops.fit_predict(X_crop_clusters)

print(f"K-Means clustering applied with {n_clusters} clusters.")
print("Crop types with their assigned clusters:")
display(crop_avg_locations.head())

# Get cluster centroids
cluster_centroids = pd.DataFrame(kmeans_crops.cluster_centers_, columns=['Latitude', 'Longitude'])
cluster_centroids['Cluster'] = range(n_clusters)

K-Means clustering applied with 5 clusters.
Crop types with their assigned clusters:


,Crop,Latitude,Longitude,Cluster
0,and gram,22.346670,77.088958,3
1,and maize,22.598055,75.303963,3
2,and mustard.,25.665317,78.460918,0
3,and oilseeds,23.832302,79.438659,0
4,and paddy,23.834344,80.389381,0


### Visualize Crop Type Clusters on a Map

In [ ]:
import pandas as pd
import folium
from sklearn.cluster import KMeans
import numpy as np

# Assuming df, expanded_df, crop_avg_locations, kmeans_crops, and cluster_centroids
# are available from previous steps.

# Create a base map
m_clusters = folium.Map(location=[20.5, 78.9], zoom_start=5)

# Define a color palette for clusters
colors = [
    'red', 'blue', 'green', 'purple', 'orange', 'darkred',
    'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
    'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen',
    'gray', 'black', 'lightgray'
]

# Merge cluster information from crop_avg_locations into expanded_df
# This assigns a cluster ID to each individual crop instance based on its crop type's cluster
df_with_clusters = expanded_df.merge(crop_avg_locations[['Crop', 'Cluster']], on='Crop', how='left')

# Add markers for each *individual crop instance*, colored by its crop type's cluster
for idx, row in df_with_clusters.iterrows():
    if pd.isna(row['Cluster']):
        continue # Skip if cluster info is missing for some reason
    cluster_color = colors[int(row['Cluster']) % len(colors)] # Cast to int
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=3, # Smaller radius for individual points
        color=cluster_color,
        fill=True,
        fill_color=cluster_color,
        fill_opacity=0.7,
        popup=f"Crop: {row['Crop'].title()}<br>Cluster: {int(row['Cluster'])}"
    ).add_to(m_clusters)

# Removed: Code to add markers for the cluster centroids

# Display the map
m_clusters